In [1]:
import pandas as pd
import os
from IPython.display import display # Para melhor visualização no Jupyter
import numpy as np
import numba

# --- CONFIGURAÇÕES ---
# 1. Defina o caminho base ONDE ESTÃO AS PASTAS 'normal' e 'ataques'
#    Ajuste este caminho conforme a localização no seu sistema.
base_project_path = os.getcwd() 
print(base_project_path)

# 3. Defina os RÓTULOS para esta categoria
label_value = 1 
attack_category_name = 'Reconnaissance'
category_path = os.path.join(base_project_path)

print(f"--- Processando Categoria: {attack_category_name} ---")
print(f"Diretório Alvo: {category_path}")
print(f"Label a ser atribuído: {label_value}")

# --- Função para ler um log Zeek ---
def read_zeek_log(file_path):
    """Lê um arquivo de log Zeek e retorna um DataFrame."""
    columns = []
    try:
        # Tenta ler com latin-1, um encoding comum para logs
        with open(file_path, 'r', encoding='latin-1') as f:
            for line in f:
                if line.startswith('#fields'):
                    columns = line.strip().split('\t')[1:]
                    break
        if not columns:
            print(f"Aviso: Não foi encontrada a linha #fields em {file_path}")
            return pd.DataFrame()

        df = pd.read_csv(
            file_path,
            sep='\t',
            names=columns,
            comment='#',
            header=None,
            low_memory=False,
            na_values=['-', '(empty)'],
            encoding='latin-1'
        )
        return df
    except FileNotFoundError:
        print(f"Erro: Arquivo não encontrado {file_path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Erro ao ler {file_path}: {e}")
        return pd.DataFrame()

# --- Processamento dos Logs ---
log_data = {} # Dicionário para guardar os DataFrames de cada tipo de log

# Lista dos tipos de log que queremos processar e colunas desejadas
log_types_info = {
    'conn': {'required': True, 'columns': None}, # Pegar todas do conn.log
    'http': {'required': False, 'columns': ['uid', 'trans_depth', 'response_body_len', 'method']},
    'ftp': {'required': False, 'columns': ['uid', 'user', 'password', 'command']},
    'dns': {'required': False, 'columns': ['uid', 'query']}
}

# Verifica se o diretório da categoria existe
if not os.path.isdir(category_path):
    print(f"ERRO CRÍTICO: Diretório não encontrado - {category_path}")
else:
    all_files_in_dir = os.listdir(category_path)

    for log_type, info in log_types_info.items():
        # Encontra todos os arquivos .log para o tipo atual usando listdir e startswith/endswith
        relevant_files = [f for f in all_files_in_dir if f.startswith(log_type + '.') and f.endswith('.log')]

        if not relevant_files:
            print(f"Nenhum arquivo {log_type}.*.log encontrado em {category_path}")
            if info['required']:
                print(f"ERRO CRÍTICO: {log_type}.log é necessário, mas não foi encontrado.")
                # exit() # Descomente para parar se conn.log não for encontrado
            log_data[log_type] = pd.DataFrame()
            continue

        # Lê cada arquivo encontrado e os concatena
        df_list = [read_zeek_log(os.path.join(category_path, f)) for f in relevant_files]
        combined_df = pd.concat(df_list, ignore_index=True)

        # Selecionar colunas específicas, se definido
        if info['columns']:
            cols_to_keep = [col for col in info['columns'] if col in combined_df.columns]
            # Garante que 'uid' esteja presente se existir no DF original
            if 'uid' not in cols_to_keep and 'uid' in combined_df.columns:
                 cols_to_keep.insert(0, 'uid')
            if not cols_to_keep or ('uid' not in cols_to_keep and len(cols_to_keep)>0) : # Se só tiver uid ou nenhuma coluna relevante
                 print(f"Aviso: Nenhuma das colunas especificadas {info['columns']} (exceto talvez uid) encontrada no {log_type}.log")
                 # Cria DF vazio apenas com UID se UID existir, senão totalmente vazio
                 if 'uid' in combined_df.columns:
                     combined_df = combined_df[['uid']].copy()
                 else:
                      combined_df = pd.DataFrame()
            elif cols_to_keep:
                 combined_df = combined_df[cols_to_keep]


        log_data[log_type] = combined_df
        print(f"Processados {len(relevant_files)} arquivo(s) {log_type}.log, total de {len(combined_df)} linhas.")


    # --- Junção (Merge) dos DataFrames ---
    if 'conn' in log_data and not log_data['conn'].empty:
        # Começa com o conn.log como base
        final_df = log_data['conn'].copy()
        print(f"\nIniciando junção com base em {len(final_df)} registros do conn.log...")

        # Junta os outros logs usando 'uid'
        for log_type in ['http', 'ftp', 'dns']:
            if log_type in log_data and not log_data[log_type].empty:
                # Garante que a coluna 'uid' exista antes de tentar o merge
                if 'uid' not in log_data[log_type].columns:
                    print(f"Aviso: DataFrame {log_type} não contém coluna 'uid'. Pulando merge.")
                    continue

                # Remove duplicatas de UID no DF secundário, mantendo a primeira ocorrência
                # (Um UID pode aparecer múltiplas vezes em http.log, por exemplo)
                log_data[log_type] = log_data[log_type].drop_duplicates(subset=['uid'], keep='first')

                # Renomeia colunas (exceto uid) para evitar conflitos
                cols_to_rename = {col: f"{log_type}_{col}" for col in log_data[log_type].columns if col != 'uid'}
                df_to_merge = log_data[log_type].rename(columns=cols_to_rename)

                final_df = pd.merge(final_df, df_to_merge, on='uid', how='left')
                print(f"Junção com {log_type}.log concluída.")
            else:
                print(f"Nenhum dado de {log_type}.log para juntar.")

        # --- Adicionar Rótulos ---
        final_df['attack_cat'] = attack_category_name
        final_df['label'] = label_value

        # --- Mostrar Resultado ---
        print("\n--- Amostra do DataFrame Final Juntado ---")
        pd.set_option('display.max_columns', None) # Garante que todas as colunas sejam mostradas
        display(final_df.head())

    else:
        print("\nERRO: Nenhum dado de conn.log encontrado ou lido com sucesso. Não é possível continuar.")

C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\reconnaissance
--- Processando Categoria: Reconnaissance ---
Diretório Alvo: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\reconnaissance
Label a ser atribuído: 1
Processados 2 arquivo(s) conn.log, total de 1966157 linhas.
Nenhum arquivo http.*.log encontrado em C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\reconnaissance
Nenhum arquivo ftp.*.log encontrado em C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\reconnaissance
Processados 1 arquivo(s) dns.log, total de 6 linhas.

Iniciando junção com base em 1966157 registros do conn.log...
Nenhum dado de http.log para juntar.
Nenhum dado de ftp.log para juntar.
Junção com dns.log concluída.

--- Amostra do DataFrame Final Juntado ---


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,dns_query,attack_cat,label
0,1.761329e+09,CDR45p4NvSAOSq20Fb,192.168.255.112,53795,192.168.255.221,443,tcp,NaN,0.000024,0.0,0.0,REJ,T,T,0,Sr,1,52,1,40,NaN,6,NaN,Reconnaissance,1
1,1.761329e+09,CexrGi10txde8HEKD9,192.168.255.112,53797,192.168.255.221,53,tcp,NaN,0.000141,0.0,0.0,REJ,T,T,0,Sr,1,52,1,40,NaN,6,NaN,Reconnaissance,1
2,1.761329e+09,CQjhQR1fPPKCUV9115,192.168.255.112,53798,192.168.255.221,554,tcp,NaN,0.000004,0.0,0.0,REJ,T,T,0,Sr,1,52,1,40,NaN,6,NaN,Reconnaissance,1
3,1.761329e+09,CICeiN2LZ4soEQEDah,192.168.255.112,53800,192.168.255.221,1025,tcp,NaN,0.000009,0.0,0.0,REJ,T,T,0,Sr,1,52,1,40,NaN,6,NaN,Reconnaissance,1
4,1.761329e+09,C5Ph3i4rRtwIqLZ8Yg,192.168.255.112,53799,192.168.255.221,111,tcp,NaN,0.000106,0.0,0.0,REJ,T,T,0,Sr,1,52,1,40,NaN,6,NaN,Reconnaissance,1


In [2]:
# --- ENGENHARIA DE FEATURES (CÁLCULOS SIMPLES) ---
# Assume que 'final_df' existe da célula anterior

print("\n--- Iniciando Engenharia de Features Simples ---")

# Verificar se o DataFrame base existe e não está vazio
if 'final_df' in locals() and not final_df.empty:

    # 1. Tratar valores numéricos que podem ser string ou NaN antes dos cálculos
    numeric_cols_to_clean = ['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts']
    for col in numeric_cols_to_clean:
        if col in final_df.columns:
            # Converte para numérico, erros viram NaN. Preenche NaN com 0.
            final_df[col] = pd.to_numeric(final_df[col], errors='coerce').fillna(0)
        else:
            print(f"Aviso: Coluna necessária '{col}' não encontrada para cálculos.")
            # Cria coluna com zeros se não existir para evitar erros posteriores
            final_df[col] = 0

    # 2. Calcular 'rate'
    # Evita divisão por zero: np.divide(..., where=denominator!=0)
    total_pkts = final_df['orig_pkts'] + final_df['resp_pkts']
    final_df['rate'] = np.divide(total_pkts, final_df['duration'], \
                                 out=np.zeros_like(total_pkts, dtype=float), where=final_df['duration']!=0)

    # 3. Calcular 'sload' (Source Load in bits per second)
    final_df['sload'] = np.divide(final_df['orig_bytes'] * 8, final_df['duration'], \
                                  out=np.zeros_like(final_df['orig_bytes'], dtype=float), where=final_df['duration']!=0)

    # 4. Calcular 'dload' (Destination Load in bits per second)
    final_df['dload'] = np.divide(final_df['resp_bytes'] * 8, final_df['duration'], \
                                  out=np.zeros_like(final_df['resp_bytes'], dtype=float), where=final_df['duration']!=0)

    # 5. Calcular 'smean' (Source Mean Packet Size)
    final_df['smean'] = np.divide(final_df['orig_bytes'], final_df['orig_pkts'], \
                                  out=np.zeros_like(final_df['orig_bytes'], dtype=float), where=final_df['orig_pkts']!=0).astype(int) # Usually integer

    # 6. Calcular 'dmean' (Destination Mean Packet Size)
    final_df['dmean'] = np.divide(final_df['resp_bytes'], final_df['resp_pkts'], \
                                  out=np.zeros_like(final_df['resp_bytes'], dtype=float), where=final_df['resp_pkts']!=0).astype(int) # Usually integer

    # 7. Calcular 'is_sm_ips_ports' (Source=Dest IP and Port)
    # Verifica se as colunas existem antes de comparar
    if 'id.orig_h' in final_df.columns and 'id.resp_h' in final_df.columns and \
       'id.orig_p' in final_df.columns and 'id.resp_p' in final_df.columns:
        final_df['is_sm_ips_ports'] = ((final_df['id.orig_h'] == final_df['id.resp_h']) & \
                                       (final_df['id.orig_p'] == final_df['id.resp_p'])).astype(int)
    else:
        print("Aviso: Colunas de IP/Porta não encontradas. 'is_sm_ips_ports' será 0.")
        final_df['is_sm_ips_ports'] = 0

    # 8. Calcular 'is_ftp_login'
    # Verifica se as colunas do FTP (resultantes do merge) existem
    if 'ftp_user' in final_df.columns and 'ftp_password' in final_df.columns:
        # Será 1 se AMBOS user e password não forem NaN (ou seja, foram preenchidos no log)
        final_df['is_ftp_login'] = ((final_df['ftp_user'].notna()) & \
                                    (final_df['ftp_password'].notna())).astype(int)
    else:
        # Se não houve merge com ftp.log ou as colunas não existiam
        print("Aviso: Colunas 'ftp_user'/'ftp_password' não encontradas. 'is_ftp_login' será 0.")
        final_df['is_ftp_login'] = 0


    print("\n--- Amostra do DataFrame Após Adicionar Features Simples ---")
    display(final_df[['uid', 'duration', 'orig_pkts', 'resp_pkts', 'orig_bytes', 'resp_bytes', \
                      'rate', 'sload', 'dload', 'smean', 'dmean', 'is_sm_ips_ports', 'is_ftp_login', \
                      'attack_cat', 'label']].head()) # Mostra apenas algumas colunas chave + as novas
    

else:
    print("\nERRO: DataFrame 'final_df' não encontrado ou vazio. Execute a célula anterior primeiro.")


--- Iniciando Engenharia de Features Simples ---
Aviso: Colunas 'ftp_user'/'ftp_password' não encontradas. 'is_ftp_login' será 0.

--- Amostra do DataFrame Após Adicionar Features Simples ---


,uid,duration,orig_pkts,resp_pkts,orig_bytes,resp_bytes,rate,sload,dload,smean,dmean,is_sm_ips_ports,is_ftp_login,attack_cat,label
0,CDR45p4NvSAOSq20Fb,0.000024,1,1,0.0,0.0,83333.333333,0.0,0.0,0,0,0,0,Reconnaissance,1
1,CexrGi10txde8HEKD9,0.000141,1,1,0.0,0.0,14184.397163,0.0,0.0,0,0,0,0,Reconnaissance,1
2,CQjhQR1fPPKCUV9115,0.000004,1,1,0.0,0.0,500000.000000,0.0,0.0,0,0,0,0,Reconnaissance,1
3,CICeiN2LZ4soEQEDah,0.000009,1,1,0.0,0.0,222222.222222,0.0,0.0,0,0,0,0,Reconnaissance,1
4,C5Ph3i4rRtwIqLZ8Yg,0.000106,1,1,0.0,0.0,18867.924528,0.0,0.0,0,0,0,0,Reconnaissance,1


In [5]:
# --- ENGENHARIA DE FEATURES (AGREGAÇÕES ct_*) ---
# Assume que 'final_df' existe das células anteriores

print("\n--- Iniciando Engenharia de Features Agregadas (ct_*) ---")

# Verificar se o DataFrame base existe e não está vazio
if 'final_df' in locals() and not final_df.empty:

    # PASSO 1: Garantir que os dados estejam ordenados por Timestamp
    print("Ordenando DataFrame por timestamp...")
    df_sorted = final_df.sort_values(by='ts').reset_index(drop=True)

    # PASSO 2: Definir o tamanho da janela
    window_size = 100
    print(f"Usando uma janela deslizante de {window_size} conexões.")

    # PASSO 3: Calcular as features ct_*

    # --- Features baseadas em IP/Porta/Serviço (Loop iterrows) ---
    results = {}
    print("Calculando features ct_* baseadas em IP/Porta/Serviço (pode levar algum tempo)...")
    if 'id.resp_h' in df_sorted.columns and 'id.resp_p' in df_sorted.columns:
        df_sorted['dst_ip_port'] = df_sorted['id.resp_h'].astype(str) + ':' + df_sorted['id.resp_p'].astype(str)
    if 'id.orig_h' in df_sorted.columns and 'id.orig_p' in df_sorted.columns:
        df_sorted['src_ip_port'] = df_sorted['id.orig_h'].astype(str) + ':' + df_sorted['id.orig_p'].astype(str)

    num_rows = len(df_sorted)
    
    for i, row in df_sorted.iterrows():
        start_idx = max(0, i - window_size + 1)
        current_window = df_sorted.iloc[start_idx : i + 1]

        # Calcula cada feature (lógica idêntica à anterior)
        if 'id.orig_h' in row and 'id.resp_p' in row and 'dst_ip_port' in row:
             results.setdefault('ct_srv_src', []).append(current_window[ (current_window['dst_ip_port'] == row['dst_ip_port']) & \
                                                                         (current_window['id.orig_h'] == row['id.orig_h']) ].shape[0])
        if 'id.resp_h' in row and 'id.orig_p' in row and 'src_ip_port' in row:
             results.setdefault('ct_srv_dst', []).append(current_window[ (current_window['src_ip_port'] == row['src_ip_port']) & \
                                                                         (current_window['id.resp_h'] == row['id.resp_h']) ].shape[0])
        if 'id.resp_h' in row:
             results.setdefault('ct_dst_ltm', []).append(current_window[ current_window['id.resp_h'] == row['id.resp_h'] ].shape[0])
        if 'id.orig_h' in row:
             results.setdefault('ct_src_ltm', []).append(current_window[ current_window['id.orig_h'] == row['id.orig_h'] ].shape[0])
        if 'id.orig_h' in row and 'id.resp_p' in row:
             results.setdefault('ct_src_dport_ltm', []).append(current_window[ (current_window['id.orig_h'] == row['id.orig_h']) & \
                                                                               (current_window['id.resp_p'] == row['id.resp_p']) ].shape[0])
        if 'id.resp_h' in row and 'id.orig_p' in row:
             results.setdefault('ct_dst_sport_ltm', []).append(current_window[ (current_window['id.resp_h'] == row['id.resp_h']) & \
                                                                               (current_window['id.orig_p'] == row['id.orig_p']) ].shape[0])
        if 'id.orig_h' in row and 'id.resp_h' in row:
             results.setdefault('ct_dst_src_ltm', []).append(current_window[ (current_window['id.orig_h'] == row['id.orig_h']) & \
                                                                             (current_window['id.resp_h'] == row['id.resp_h']) ].shape[0])

        if (i + 1) % 500 == 0:
            print(f"  Processado {i + 1}/{num_rows} linhas...")

    print("  Cálculo das features baseadas em IP/Porta/Serviço concluído.")
    for feature_name, values in results.items():
         if len(values) == len(df_sorted):
              df_sorted[feature_name] = values
         else:
              print(f"Erro de tamanho: '{feature_name}'. Preenchendo com 0.")
              df_sorted[feature_name] = 0
    df_sorted = df_sorted.drop(columns=['dst_ip_port', 'src_ip_port'], errors='ignore')

    # -- Features Específicas (ct_state_ttl, ct_ftp_cmd, ct_flw_http_mthd) --

    # ct_state_ttl (Contagem por estado) - Usando Factorization e indexação NumPy
    print("Calculando ct_state_ttl (usando factorization)...")
    if 'conn_state' in df_sorted.columns:
        state_codes, state_uniques = pd.factorize(df_sorted['conn_state'])
        df_sorted['_state_codes'] = state_codes

        try:
             # CORREÇÃO NA LAMBDA: Usa x[-1] em vez de x.iloc[-1]
             ct_state_ttl_result = df_sorted.groupby('conn_state')['_state_codes'] \
                                            .rolling(window=window_size, min_periods=1) \
                                            .apply(lambda x: (x == x[-1]).sum(), raw=True, engine='numba') # <- Correção
        except Exception:
             print("  Numba não disponível ou falhou, usando engine padrão...")
             # CORREÇÃO NA LAMBDA: Usa x[-1] em vez de x.iloc[-1]
             ct_state_ttl_result = df_sorted.groupby('conn_state')['_state_codes'] \
                                            .rolling(window=window_size, min_periods=1) \
                                            .apply(lambda x: (x == x[-1]).sum(), raw=True) # <- Correção

        ct_state_ttl_result = ct_state_ttl_result.reset_index(level=0, drop=True) \
                                                 .sort_index() \
                                                 .fillna(1).astype(int)
        df_sorted['ct_state_ttl'] = ct_state_ttl_result
        df_sorted = df_sorted.drop(columns=['_state_codes'])
    else:
        print("Aviso: Coluna 'conn_state' não encontrada. 'ct_state_ttl' será 0.")
        df_sorted['ct_state_ttl'] = 0

    # ct_ftp_cmd (Lógica idêntica à anterior)
    print("Calculando ct_ftp_cmd...")
    if 'ftp_command' in df_sorted.columns:
        df_sorted['ct_ftp_cmd'] = df_sorted['ftp_command'].notna().rolling(window=window_size, min_periods=1).sum().fillna(0).astype(int)
    else:
        print("Aviso: Coluna 'ftp_command' não encontrada. 'ct_ftp_cmd' será 0.")
        df_sorted['ct_ftp_cmd'] = 0

    # ct_flw_http_mthd (Lógica idêntica à anterior)
    print("Calculando ct_flw_http_mthd...")
    if 'http_method' in df_sorted.columns:
        df_sorted['ct_flw_http_mthd'] = df_sorted['http_method'].notna().rolling(window=window_size, min_periods=1).sum().fillna(0).astype(int)
    else:
        print("Aviso: Coluna 'http_method' não encontrada. 'ct_flw_http_mthd' será 0.")
        df_sorted['ct_flw_http_mthd'] = 0

    print("\n--- Amostra do DataFrame Após Adicionar Features Agregadas ---")
    ct_cols_to_show = [col for col in df_sorted.columns if col.startswith('ct_')]
    display(df_sorted[['ts', 'uid', 'id.orig_h', 'id.resp_h'] + ct_cols_to_show].head())
    display(df_sorted[['ts', 'uid', 'id.orig_h', 'id.resp_h'] + ct_cols_to_show].tail())

    final_engineered_df = df_sorted # Atribui o resultado final
    # --- Exportar para CSV (Opcional) ---
    output_filename = f"{attack_category_name.lower().replace(os.path.sep, '_')}_processed.csv" # Nomeia arquivo baseado na categoria
    final_engineered_df.to_csv(os.path.join(category_path, output_filename), index=False)
    print(f"\nDataFrame exportado para {output_filename}")

else:
    print("\nERRO: DataFrame 'final_df' não encontrado ou vazio. Execute as células anteriores primeiro.")


--- Iniciando Engenharia de Features Agregadas (ct_*) ---
Ordenando DataFrame por timestamp...
Usando uma janela deslizante de 100 conexões.
Calculando features ct_* baseadas em IP/Porta/Serviço (pode levar algum tempo)...
  Processado 500/1966157 linhas...
  Processado 1000/1966157 linhas...
  Processado 1500/1966157 linhas...
  Processado 2000/1966157 linhas...
  Processado 2500/1966157 linhas...
  Processado 3000/1966157 linhas...
  Processado 3500/1966157 linhas...
  Processado 4000/1966157 linhas...
  Processado 4500/1966157 linhas...
  Processado 5000/1966157 linhas...
  Processado 5500/1966157 linhas...
  Processado 6000/1966157 linhas...
  Processado 6500/1966157 linhas...
  Processado 7000/1966157 linhas...
  Processado 7500/1966157 linhas...
  Processado 8000/1966157 linhas...
  Processado 8500/1966157 linhas...
  Processado 9000/1966157 linhas...
  Processado 9500/1966157 linhas...
  Processado 10000/1966157 linhas...
  Processado 10500/1966157 linhas...
  Processado 11000/

,ts,uid,id.orig_h,id.resp_h,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_state_ttl,ct_ftp_cmd,ct_flw_http_mthd
0,1.761329e+09,Ck5Vxw1VsUk74Q4Pdf,192.168.255.112,192.168.255.221,1,1,1,1,1,1,1,1,0,0
1,1.761329e+09,CDR45p4NvSAOSq20Fb,192.168.255.112,192.168.255.221,1,1,2,2,1,1,2,1,0,0
2,1.761329e+09,Chwe6i1mX0Ymv0bEAh,192.168.255.112,192.168.255.221,1,1,3,3,1,1,3,2,0,0
3,1.761329e+09,CexrGi10txde8HEKD9,192.168.255.112,192.168.255.221,1,1,4,4,1,1,4,2,0,0
4,1.761329e+09,C5Ph3i4rRtwIqLZ8Yg,192.168.255.112,192.168.255.221,1,1,5,5,1,1,5,3,0,0


,ts,uid,id.orig_h,id.resp_h,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_state_ttl,ct_ftp_cmd,ct_flw_http_mthd
1966152,1.761330e+09,C66On94h1VFpvD0SNf,192.168.255.221,192.168.255.1,2,1,2,2,2,1,2,4,0,0
1966153,1.761330e+09,CJFAuB1pKMOZ0LDaYb,192.168.255.221,54.217.10.153,1,1,1,3,1,1,1,100,0,0
1966154,1.761330e+09,CqewFm3BfjaOJo395c,192.168.255.221,54.217.10.153,2,2,2,4,2,2,2,1,0,0
1966155,1.761330e+09,CJaCej1l352rObOLCj,192.168.255.118,239.255.255.250,1,1,1,1,1,1,1,17,0,0
1966156,1.761330e+09,CjT6wGqfwp7fy2fO9,192.168.255.118,239.255.255.250,1,1,2,2,1,1,2,18,0,0



DataFrame exportado para reconnaissance_processed.csv
